In [1]:
from pathlib import Path

BASE_DIR = Path.cwd().parent.parent

import configparser

def load_config(filename=BASE_DIR / "app/config.ini"):
    """Parses the config.ini file into a usable dictionary."""
    # Use inline_comment_prefixes to ignore everything after '#'
    config_parser = configparser.ConfigParser(inline_comment_prefixes=('#',))
    config_parser.read(filename)

    c = {}

    # PRICE ARGUMENTS Section
    price = config_parser['PRICE_ARGUMENTS']
    c['noise_scale'] = float(price['noise_scale'])
    c['base'] = float(price['base'])
    c['room_coeff'] = float(price['room_coeff'])
    c['distance_coeff'] = float(price['distance_coeff'])
    c['underground_fee'] = float(price['underground_fee'])
    c['house_fee'] = float(price['house_fee'])
    c['garden_fee'] = float(price['garden_fee'])
    c['terrace_fee'] = float(price['terrace_fee'])
    c['balcony_fee'] = float(price['balcony_fee'])
    c['agent_premium'] = float(price['agent_premium'])

    return c

config = load_config()

In [ ]:
import os

# Disable the C-linker to avoid the relocation/linkage error.
# This will use the Python backend.
os.environ["PYTENSOR_FLAGS"] = "cxx=,linker=py"

import pandas as pd
import numpy as np
import scipy.stats as stats

# Load the generated dataset
df = pd.read_csv("../../data/datasets/london_rentals_fixed.csv")

# --- DEFINING THE "TRUTH" ---
INTERCEPT = config['base']
BETA_ROOM = config['room_coeff']
BETA_DIST = config['distance_coeff']
BETA_UNDERGROUND = config['underground_fee']
SIGMA = config['noise_scale']
AGENT_PREMIUM = config['agent_premium']

# Mappings for categorical logic
TYPE_MAP = {'House': config['house_fee'], 'Flat': 0.0}
OUTDOOR_MAP = {'Garden': config['garden_fee'], 'Terrace': config['terrace_fee'], 'Balcony': config['balcony_fee'], 'Nothing': 0.0}

# --- FEATURE ENGINEERING ---
# We transform everything into (natural) log-space because the math is additive there.
df['log_rent'] = np.log(df['monthly_rent_gbp'])

# Calculate the intrinsic value of the property (everything except the Agent markup)
# This is our mu (mean) for the 'Owner' hypothesis.
df['expected_log_rent_base'] = (
    INTERCEPT + 
    (df['n_rooms'] * BETA_ROOM) +
    (df['dist_centre_km'] * BETA_DIST) +
    (df['near_underground'] * BETA_UNDERGROUND) +
    df['property_type'].map(TYPE_MAP) +
    df['outdoor_space'].map(OUTDOOR_MAP)
)

print(f"Dataset loaded. Total properties: {len(df)}")
print(f"Reference Sigma: {SIGMA} (approx {SIGMA*100}% price fluctuation)")
df[['monthly_rent_gbp', 'log_rent', 'expected_log_rent_base']].head()

Dataset loaded. Total properties: 1000
Reference Sigma: 0.05 (approx 5.0% price fluctuation)


,monthly_rent_gbp,log_rent,expected_log_rent_base
0,4712.827954,8.458043,8.380476
1,3418.436071,8.136938,8.132979
2,1832.056346,7.513194,7.504495
3,713.112913,6.569640,6.573257
4,2636.099044,7.877055,7.977272


In [3]:
# --- BAYESIAN LIKELIHOOD CALCULATION ---

"""
Our generative model defined price as: 
    ln(Price) = expected_log_rent + Noise(0, sigma^2)

To perform inference, we calculate the 'Likelihood'—the probability of observing 
the actual rent given a specific hypothesis (Owner vs. Agent). 

If the noise is Gaussian, the likelihood follows the Normal PDF:
    P(Data | Hypothesis) = (1 / (σ * sqrt(2π))) * exp(-0.5 * ((x - μ) / σ)^2)

Where:
    x = The observed 'log_rent'
    μ = Our calculated 'expected_log_rent_base' (plus premium if Agent)
    σ = The 'SIGMA' representing the noise
"""

# Hypothesis A: The property is an Owner listing (No Premium)
# Here, μ = expected_log_rent_base
df['likelihood_owner'] = stats.norm.pdf(
    df['log_rent'], 
    loc=df['expected_log_rent_base'], 
    scale=SIGMA
)

# Hypothesis B: The property is an Agent listing (Includes Premium)
# Here, μ = expected_log_rent_base + AGENT_PREMIUM
df['likelihood_agent'] = stats.norm.pdf(
    df['log_rent'], 
    loc=df['expected_log_rent_base'] + AGENT_PREMIUM, 
    scale=SIGMA
)

# Preview the results
print("Likelihoods calculated based on Normal PDF of residuals.")
df[['monthly_rent_gbp', 'listing_type', 'likelihood_owner', 'likelihood_agent']].head()

Likelihoods calculated based on Normal PDF of residuals.


,monthly_rent_gbp,listing_type,likelihood_owner,likelihood_agent
0,4712.827954,Owner,2.395201,2.794029
1,3418.436071,Owner,7.953869,0.112053
2,1832.056346,Owner,7.859004,0.147134
3,713.112913,Owner,7.957998,0.071159
4,2636.099044,Owner,1.070500,0.000029


In [4]:
# --- MODEL COMPARISON: POSTERIOR ODDS RATIO ---

# Calculate the Bayes Factor (Likelihood Ratio)
# How much better does the Agent model explain the price than the Owner model?
df['bayes_factor'] = df['likelihood_agent'] / df['likelihood_owner']

# Prior Odds (0.5 / 0.5 = 1)
prior_odds = 1.0

# Posterior Odds Ratio
df['posterior_odds'] = df['bayes_factor'] * prior_odds

# Decision Logic
# Ratio > 1 -> Evidence favors Agent
# Ratio < 1 -> Evidence favors Owner
print("Odds Ratios calculated. Evidence check:")
df[['monthly_rent_gbp', 'listing_type', 'bayes_factor', 'posterior_odds']].head(10)

Odds Ratios calculated. Evidence check:


,monthly_rent_gbp,listing_type,bayes_factor,posterior_odds
0,4712.827954,Owner,1.166511,1.166511
1,3418.436071,Owner,0.014088,0.014088
2,1832.056346,Owner,0.018722,0.018722
3,713.112913,Owner,0.008942,0.008942
4,2636.099044,Owner,0.000027,0.000027
5,3077.518013,Owner,0.017146,0.017146
6,2643.429601,Owner,0.000123,0.000123
7,4840.155235,Owner,0.020928,0.020928
8,1802.483564,Owner,0.002082,0.002082
9,3234.415378,Owner,0.287534,0.287534


In [5]:
# --- ACCURACY EVALUATION ---

# Pick the model with the higher odds
df['predicted_type'] = np.where(df['posterior_odds'] > 1, 'Agent', 'Owner')

# Check if the prediction matches the ground truth
df['is_correct'] = df['predicted_type'] == df['listing_type']

# Calculate overall accuracy percentage
accuracy = df['is_correct'].mean() * 100

print(f"--- Model Accuracy: {accuracy:.2f}% ---")

# Breakdown by class
print("\nPrediction Breakdown:")
print(pd.crosstab(df['listing_type'], df['predicted_type']))

--- Model Accuracy: 93.50% ---

Prediction Breakdown:
predicted_type  Agent  Owner
listing_type                
Agent             466     34
Owner              31    469
